In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import warnings
from datetime import datetime
import time

warnings.filterwarnings('ignore')

# ============================================================================
# 1. CONFIGURACIÓN INICIAL
# ============================================================================

# Tickers por sector estratégico
TICKERS_CONFIG = {
    'Retail Tradicional': ['WMT', 'TGT', 'COST', 'HD'],
    'Automotriz': ['F', 'GM', 'TSLA'],
    'Fast Fashion': ['ANF', 'GAP', 'URBN']
}

# Rango de años objetivo
AÑOS_OBJETIVO = list(range(2015, 2026))  # 2015-2025

# Mapeo de variables financieras (aliases de Yahoo Finance)
BALANCE_SHEET_MAPPING = {
    'cash_equivalents': [
        'Cash And Cash Equivalents',
        'Cash',
        'Cash And Cash Equivalents At Carrying Value'
    ],
    'accounts_receivable': [
        'Current Assets - Receivables',
        'Accounts Receivable',
        'Current Receivables'
    ],
    'inventory': [
        'Inventory',
        'Inventories'
    ],
    'current_assets': [
        'Current Assets',
        'Total Current Assets'
    ],
    'current_liabilities': [
        'Current Liabilities',
        'Total Current Liabilities'
    ]
}

INCOME_STATEMENT_MAPPING = {
    'total_revenue': [
        'Total Revenue',
        'Net Revenue',
        'Total Net Revenue'
    ],
    'cost_of_revenue': [
        'Cost of Revenue',
        'Cost of Goods Sold',
        'COGS'
    ]
}

# ============================================================================
# 2. FUNCIONES AUXILIARES
# ============================================================================

def descargar_datos_empresa(ticker, max_intentos=3):
    """
    Descarga estados financieros anuales de una empresa

    Args:
        ticker (str): Símbolo del ticker
        max_intentos (int): Número de reintentos en caso de error

    Returns:
        tuple: (DataFrame balance_sheet, DataFrame income_statement) o (None, None)
    """
    for intento in range(max_intentos):
        try:
            empresa = yf.Ticker(ticker)

            # Descarga balance sheet y state of income (anuales)
            bs = empresa.balance_sheet
            fs = empresa.income_stmt

            if bs is None or fs is None or bs.empty or fs.empty:
                print(f"⚠️  {ticker}: Datos vacíos en intento {intento + 1}")
                time.sleep(2)
                continue

            print(f"✓ {ticker}: Descargado exitosamente")
            return bs, fs

        except Exception as e:
            print(f"⚠️  {ticker}: Error en intento {intento + 1} - {str(e)}")
            time.sleep(2)

    print(f"✗ {ticker}: No se pudo descargar después de {max_intentos} intentos")
    return None, None


def extraer_valor_financiero(df, alias_list, año):
    """
    Extrae valor de una variable financiera buscando en lista de alias
    Busca por la columna más cercana al año objetivo (tolerancia ±1 año)

    Args:
        df (DataFrame): Estado financiero (balance sheet o income statement)
        alias_list (list): Lista de posibles nombres de la variable
        año (int): Año buscado

    Returns:
        float: Valor extraído o NaN si no se encuentra
    """
    if df is None or df.empty:
        return np.nan

    # Busca por alias
    fila = None
    for alias in alias_list:
        if alias in df.index:
            fila = df.loc[alias]
            break

    if fila is None:
        return np.nan

    # Busca fecha más cercana al año objetivo
    # Yahoo Finance organiza por datetime, necesitamos extraer el año
    fechas_disponibles = []
    for col in fila.index:
        if isinstance(col, pd.Timestamp):
            fechas_disponibles.append(col)

    if not fechas_disponibles:
        return np.nan

    # Ordena por proximidad al año objetivo
    año_dt = pd.Timestamp(f'{año}-12-31')
    col_cercana = min(fechas_disponibles, key=lambda x: abs((x - año_dt).days))

    # Tolerancia: máximo 1 año de diferencia
    if abs((col_cercana - año_dt).days) > 365:
        return np.nan

    valor = fila[col_cercana]
    return float(valor) if pd.notna(valor) else np.nan


def procesar_empresa(ticker, sector):
    """
    Procesa datos financieros de una empresa y calcula ratios de WC

    Args:
        ticker (str): Símbolo del ticker
        sector (str): Sector al que pertenece

    Returns:
        DataFrame: Datos procesados y ratios calculados
    """
    print(f"\n{'='*60}")
    print(f"Procesando: {ticker} ({sector})")
    print(f"{'='*60}")

    # Descarga datos
    bs, fs = descargar_datos_empresa(ticker)
    if bs is None or fs is None:
        return None

    registros = []

    for año in AÑOS_OBJETIVO:
        registro = {
            'ticker': ticker,
            'sector': sector,
            'año': año
        }

        # === BALANCE SHEET ===
        cash = extraer_valor_financiero(bs, BALANCE_SHEET_MAPPING['cash_equivalents'], año)
        ar = extraer_valor_financiero(bs, BALANCE_SHEET_MAPPING['accounts_receivable'], año)
        inv = extraer_valor_financiero(bs, BALANCE_SHEET_MAPPING['inventory'], año)
        ca = extraer_valor_financiero(bs, BALANCE_SHEET_MAPPING['current_assets'], año)
        cl = extraer_valor_financiero(bs, BALANCE_SHEET_MAPPING['current_liabilities'], año)

        # === INCOME STATEMENT ===
        revenue = extraer_valor_financiero(fs, INCOME_STATEMENT_MAPPING['total_revenue'], año)
        cogs = extraer_valor_financiero(fs, INCOME_STATEMENT_MAPPING['cost_of_revenue'], año)

        # Almacena valores brutos
        registro['cash'] = cash
        registro['accounts_receivable'] = ar
        registro['inventory'] = inv
        registro['current_assets'] = ca
        registro['current_liabilities'] = cl
        registro['revenue'] = revenue
        registro['cogs'] = cogs

        registros.append(registro)

    df_empresa = pd.DataFrame(registros)
    return df_empresa


def calcular_ratios(df):
    """
    Calcula ratios de gestión de working capital
    Maneja correctamente divisiones por cero y valores NaN

    Args:
        df (DataFrame): DataFrame con datos brutos

    Returns:
        DataFrame: DataFrame con 10 columnas finales
    """
    df = df.copy()

    # ===== PORCENTAJES DE ACTIVOS CORRIENTES =====
    # Evita división por cero si current_assets es 0 o NaN
    df['cash_pct'] = np.where(
        (df['current_assets'] != 0) & (df['current_assets'].notna()),
        (df['cash'] / df['current_assets'] * 100).round(2),
        np.nan
    )

    df['receivables_pct'] = np.where(
        (df['current_assets'] != 0) & (df['current_assets'].notna()),
        (df['accounts_receivable'] / df['current_assets'] * 100).round(2),
        np.nan
    )

    df['inventory_pct'] = np.where(
        (df['current_assets'] != 0) & (df['current_assets'].notna()),
        (df['inventory'] / df['current_assets'] * 100).round(2),
        np.nan
    )

    # ===== CICLO DE CONVERSIÓN DE EFECTIVO =====
    # DSO: evita división por cero si revenue es 0 o NaN
    df['dso'] = np.where(
        (df['revenue'] != 0) & (df['revenue'].notna()),
        ((df['accounts_receivable'] / df['revenue']) * 365).round(2),
        np.nan
    )

    # DIO y DPO: Si COGS no está disponible, usa revenue como aproximación
    cogs_valido = df['cogs'].copy().to_numpy() # Convert to numpy array explicitly
    cogs_valido = np.where(
        (cogs_valido == 0) | (np.isnan(cogs_valido)), # Use np.isnan for numpy arrays
        df['revenue'].to_numpy(),  # Usa revenue como fallback, también como numpy array
        cogs_valido
    )

    df['dio'] = np.where(
        (cogs_valido != 0) & (~np.isnan(cogs_valido)), # Use ~np.isnan for numpy arrays
        ((df['inventory'].to_numpy() / cogs_valido) * 365).round(2), # Ensure operations are on numpy arrays
        np.nan
    )

    df['dpo'] = np.where(
        (cogs_valido != 0) & (~np.isnan(cogs_valido)), # Use ~np.isnan for numpy arrays
        ((df['current_liabilities'].to_numpy() / cogs_valido) * 365).round(2), # Ensure operations are on numpy arrays
        np.nan
    )

    # CCC: suma de DSO + DIO - DPO
    df['ccc'] = (df['dso'] + df['dio'] - df['dpo']).round(2)

    # Mantiene solo las 10 columnas finales
    df_final = df[['ticker', 'sector', 'año', 'cash_pct', 'receivables_pct',
                   'inventory_pct', 'dso', 'dio', 'dpo', 'ccc']].copy()

    return df_final


def imputar_datos(df, columnas_imputar):
    """
    Imputa valores faltantes en un DataFrame.
    Primero usa la media por empresa, luego la media global si persisten NaN.

    Args:
        df (DataFrame): DataFrame con datos faltantes
        columnas_imputar (list): Columnas a imputar

    Returns:
        DataFrame: DataFrame con imputación realizada
    """
    df = df.copy()

    # Primera pasada: Imputar usando la media por empresa (ticker)
    for ticker in df['ticker'].unique():
        mask_ticker = df['ticker'] == ticker

        for col in columnas_imputar:
            media_ticker = df.loc[mask_ticker, col].mean()

            if not pd.isna(media_ticker):
                df.loc[mask_ticker & df[col].isna(), col] = media_ticker

    # Segunda pasada: Imputar los NaN restantes usando la media global de la columna
    for col in columnas_imputar:
        if df[col].isna().any():
            media_global = df[col].mean()
            if not pd.isna(media_global):
                df[col].fillna(media_global, inplace=True)

    return df


def validar_rango_años(df):
    """
    Valida que los datos estén en el rango 2015-2025

    Args:
        df (DataFrame): DataFrame a validar

    Returns:
        DataFrame: DataFrame filtrado
    """
    df_valido = df[(df['año'] >= 2015) & (df['año'] <= 2025)].copy()

    filas_eliminadas = len(df) - len(df_valido)
    if filas_eliminadas > 0:
        print(f"⚠️  Se eliminaron {filas_eliminadas} filas fuera del rango 2015-2025")

    return df_valido


# ============================================================================
# 3. PIPELINE PRINCIPAL
# ============================================================================

print("\n" + "="*70)
print("INICIANDO DESCARGA DE DATOS FINANCIEROS - ML WORKING CAPITAL")
print("="*70)
print(f"Fecha de ejecución: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Años objetivo: 2015-2025")
print(f"Empresas: 10 | Sectores: 3")
print("="*70)

# Lista para almacenar DataFrames de todas las empresas
dfs_empresas = []

# Itera sobre sectores y tickers
for sector, tickers in TICKERS_CONFIG.items():
    print(f"\n\n{'#'*70}")
    print(f"SECTOR: {sector.upper()}")
    print(f"{'#'*70}")

    for ticker in tickers:
        df_empresa = procesar_empresa(ticker, sector)

        if df_empresa is not None:
            dfs_empresas.append(df_empresa)
        else:
            print(f"✗ Se omitió {ticker} por errores de descarga")

# Combina todos los datos
if dfs_empresas:
    df_maestro = pd.concat(dfs_empresas, ignore_index=True)
    print(f"\n✓ Combinadas {len(dfs_empresas)} empresas")
else:
    print("\n✗ ERROR: No se pudieron descargar datos de ninguna empresa")
    raise ValueError("Pipeline fallido: Sin datos disponibles")

# ============================================================================
# 4. LIMPIEZA E INGENIERÍA DE VARIABLES
# ============================================================================

print("\n" + "="*70)
print("LIMPIEZA E INGENIERÍA DE VARIABLES")
print("="*70)

# Valida rango de años
df_maestro = validar_rango_años(df_maestro)
print(f"✓ Filas tras validación de años: {len(df_maestro)}")

# Columnas a imputar (variables financieras)
columnas_basicas = [
    'cash', 'accounts_receivable', 'inventory',
    'current_assets', 'current_liabilities',
    'revenue', 'cogs'
]

# Imputa valores faltantes
print(f"\nImputando valores faltantes por empresa...")
missing_antes = df_maestro[columnas_basicas].isna().sum().sum()
df_maestro = imputar_datos(df_maestro, columnas_basicas)
missing_después = df_maestro[columnas_basicas].isna().sum().sum()
print(f"✓ NaN antes: {missing_antes} | NaN después: {missing_después}")

# Calcula ratios después de la imputación
df_maestro = calcular_ratios(df_maestro)
print(f"✓ Ratios calculados y dataset reducido a 10 columnas")

# ============================================================================
# 5. REPORTE FINAL Y ESTADÍSTICAS
# ============================================================================

print("\n" + "="*70)
print("RESUMEN ESTADÍSTICO FINAL")
print("="*70)

print(f"\nDimensiones del dataset:")
print(f"  • Filas: {len(df_maestro)}")
print(f"  • Columnas: {len(df_maestro.columns)}")
print(f"\nCobertura por sector:")
sector_counts = df_maestro.groupby('sector').size()
for sector, count in sector_counts.items():
    print(f"  • {sector}: {count} registros")

print(f"\nCobertura temporal:")
print(f"  • Rango: {df_maestro['año'].min()} - {df_maestro['año'].max()}")
print(f"  • Años únicos: {sorted(df_maestro['año'].unique())}")

print(f"\nEstadísticos de variables clave:")
print(df_maestro[['cash_pct', 'receivables_pct', 'inventory_pct', 'dso', 'dio', 'dpo', 'ccc']].describe().round(2))

# ============================================================================
# 6. VISUALIZACIÓN Y DESCARGA
# ============================================================================

print("\n" + "="*70)
print("PRIMERAS FILAS DEL DATASET")
print("="*70)
print(df_maestro.head(15).to_string())

print("\n" + "="*70)
print("ESTRUCTURA DEL DATASET")
print("="*70)
print(df_maestro.info())

# Descarga en Google Colab
try:
    from google.colab import files

    print("\n" + "="*70)
    print("DESCARGANDO ARCHIVO A TU PC")
    print("="*70)

    # Guarda CSV en directorio temporal
    nombre_archivo = 'dataset_comps_anual.csv'
    df_maestro.to_csv(nombre_archivo, index=False, encoding='utf-8')
    print(f"✓ Archivo guardado: {nombre_archivo}")

    # Descarga
    files.download(nombre_archivo)
    print(f"✓ Descarga iniciada: {nombre_archivo}")

except ImportError:
    print("\n⚠️  No estás en Google Colab, guardando CSV localmente...")
    nombre_archivo = 'dataset_comps_anual.csv'
    df_maestro.to_csv(nombre_archivo, index=False, encoding='utf-8')
    print(f"✓ Archivo guardado: {nombre_archivo}")

print("\n" + "="*70)
print("✓ PIPELINE COMPLETADO EXITOSAMENTE")
print("="*70)
print(f"\nTiempo final: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Dataset listo para análisis de clustering y ML predictivo")

# Retorna el DataFrame maestro para uso posterior
print("\nDataFrame disponible como: df_maestro")


INICIANDO DESCARGA DE DATOS FINANCIEROS - ML WORKING CAPITAL
Fecha de ejecución: 2026-05-21 03:03:04
Años objetivo: 2015-2025
Empresas: 10 | Sectores: 3


######################################################################
SECTOR: RETAIL TRADICIONAL
######################################################################

Procesando: WMT (Retail Tradicional)
✓ WMT: Descargado exitosamente

Procesando: TGT (Retail Tradicional)
✓ TGT: Descargado exitosamente

Procesando: COST (Retail Tradicional)
✓ COST: Descargado exitosamente

Procesando: HD (Retail Tradicional)
✓ HD: Descargado exitosamente


######################################################################
SECTOR: AUTOMOTRIZ
######################################################################

Procesando: F (Automotriz)
✓ F: Descargado exitosamente

Procesando: GM (Automotriz)
✓ GM: Descargado exitosamente

Procesando: TSLA (Automotriz)
✓ TSLA: Descargado exitosamente


#######################################################

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Descarga iniciada: dataset_comps_anual.csv

✓ PIPELINE COMPLETADO EXITOSAMENTE

Tiempo final: 2026-05-21 03:03:06
Dataset listo para análisis de clustering y ML predictivo

DataFrame disponible como: df_maestro
